# Demo 4 · Putting it all together: Titanic

On 15 April 1912, the *Titanic* sank after hitting an iceberg; more than 1,500 of the
2,224 passengers and crew died. Kaggle's
[Titanic competition](https://www.kaggle.com/c/titanic) asks a simple question:
**from what we know about a passenger, can we predict whether they survived?**

This notebook walks through a complete data-science project on that question, reusing
the tools from Demo 1 and from notebooks
[01](01_validation_and_hyperparameter_search.ipynb) and
[02](02_bootstrap_confidence_intervals.ipynb):

```text
question → data → explore → clean and engineer features → model → validate → predict and share
```

By the end, you will be able to:

- Inspect a new dataset: its columns, its target, its missing values.
- Use plots to find which variables are related to the target.
- Turn raw columns (text, categories, missing values) into numeric features in a reusable function.
- Compare several classifiers, combine them in an ensemble, and validate the result with cross-validation.
- Produce a prediction file in the format Kaggle expects.

### Exercises
Six exercises are placed where they fit in the project. Each has an empty cell for your
work and a reference answer to open **after** trying.

| | Exercise | Section |
|---|---|---|
| Q1 | Impute the missing ages | D · Feature 4 |
| Q2 | Engineer features from `Fare`, `Name` and `Ticket` | D · Features 7–9 |
| Q3 | Try PCA on the features | E |
| Q4 | Tune a model with `RandomizedSearchCV` | I |
| Q5 | Reach the best accuracy you can | I |
| Q6 | A bootstrap confidence interval for the final model | I |

## Setup

In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "numpy>=1.26",
        "pandas>=2.2,<4",
        "matplotlib>=3.7",
        "seaborn>=0.13,<0.14",
        "scikit-learn>=1.4",
        "scipy>=1.11",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import (
    AdaBoostClassifier,
    ExtraTreesClassifier,
    RandomForestClassifier,
    VotingClassifier,
)
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/milarobotlearningcourse/data_science/main/demo_4/datasets/"

# Locally, find demo_4/datasets by walking up from the notebook folder.
ROOT, DATA_DIR = Path.cwd(), None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "demo_4" / "datasets").is_dir():
        ROOT, DATA_DIR = folder, folder / "demo_4" / "datasets"
        break


def data_source(filename):
    """Return the local CSV path if the repository is available, otherwise its URL."""
    if DATA_DIR is not None:
        return DATA_DIR / filename
    return DATA_URL + filename


print("Reading data from:", DATA_DIR or DATA_URL)

## A. The data

Kaggle provides two files:

- `titanic-train.csv`: 891 passengers **with** the answer (`Survived`). We learn from it.
- `titanic-test.csv`: 418 other passengers **without** the answer. Kaggle keeps their
  outcomes secret and scores the predictions we submit.

We keep the files as loaded in `train_raw` and `test_raw` and never modify them, so we can
always start again from the original data.

In [ ]:
train_raw = pd.read_csv(data_source("titanic-train.csv"))
test_raw = pd.read_csv(data_source("titanic-test.csv"))
print("train:", train_raw.shape, "| test:", test_raw.shape)
train_raw.info()

In [ ]:
display(train_raw.head())

### What the columns mean

| Column | Meaning |
|---|---|
| `PassengerId` | Row identifier |
| `Survived` | 1 = survived, 0 = died (**the target**, absent from the test file) |
| `Pclass` | Ticket class: 1 = first (upper), 2 = second (middle), 3 = third (lower); a proxy for socio-economic status |
| `Name` | Full name, including a title such as *Mr*, *Mrs*, *Miss* or *Master* |
| `Sex` | `male` or `female` |
| `Age` | Age in years; fractional below 1, and of the form *xx.5* when estimated |
| `SibSp` | Number of siblings or spouses aboard |
| `Parch` | Number of parents or children aboard (some children travelled with a nanny only, so `Parch` = 0) |
| `Ticket` | Ticket number |
| `Fare` | Ticket price (in 1912 pounds) |
| `Cabin` | Cabin number, often missing |
| `Embarked` | Port of embarkation: C = Cherbourg, Q = Queenstown, S = Southampton |

**Check yourself:** how many feature variables could we use, and which column is the target?

<details>
<summary>Answer</summary>

`Survived` is the target. `PassengerId` is only an identifier, so it carries no information
about survival. That leaves **10 candidate features**: `Pclass`, `Name`, `Sex`, `Age`, `SibSp`,
`Parch`, `Ticket`, `Fare`, `Cabin` and `Embarked`. Several of them (`Name`, `Ticket`, `Cabin`)
are text and need work before a model can use them.

</details>

## B. Missing values

Models in scikit-learn do not accept missing values, so we need to know where they are,
in **both** files.

In [ ]:
missing = pd.DataFrame({
    "train": train_raw.isna().sum(),
    "test": test_raw.isna().sum(),
})
display(missing[missing.sum(axis=1) > 0])

- `Age` is missing for about 20% of passengers: worth filling in carefully (exercise Q1).
- `Cabin` is missing for about 77%: too many to fill in, but *having* a recorded cabin may itself be informative.
- `Embarked` is missing twice in the training file; we will use the most frequent port.
- `Fare` is missing once, **only in the test file**. Any feature built from `Fare` must handle that row, or predicting on the test file will fail.

## C. Exploring the data

### How many passengers survived?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.countplot(data=train_raw, x="Survived", ax=axes[0])
axes[0].set_title("Number of passengers")
(train_raw["Survived"].value_counts(normalize=True).sort_index() * 100).plot(kind="bar", ax=axes[1])
axes[1].set_title("Share of passengers (%)")
axes[1].set_xlabel("Survived")
plt.tight_layout()
plt.show()

About 38% of passengers in the training file survived. A model that always predicts
"died" would therefore be right 62% of the time: **every model must beat this baseline**.

### Age and survival

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=train_raw, x="Age", hue="Survived", bins=20, ax=axes[0])
axes[0].set_title("Age distribution (counts)")
sns.kdeplot(data=train_raw, x="Age", hue="Survived", fill=True, common_norm=False, ax=axes[1])
axes[1].set_title("Age distribution (density within each group)")
plt.tight_layout()
plt.show()

The density plot rescales each group so their shapes can be compared. Young children
are more frequent among survivors; most other ages look similar in both groups.

### Outliers
Box plots show the median, the middle 50% of values (the box) and points far from the rest.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=train_raw, x="Survived", y="Age", hue="Sex", ax=axes[0])
sns.boxplot(data=train_raw, x="Survived", y="Fare", hue="Sex", ax=axes[1])
plt.tight_layout()
plt.show()

A few fares exceed £500 while the median is about £14. Such a skewed variable is often
easier for models after a log transform (an idea for Q2).

### Correlations between numeric columns
`numeric_only=True` ignores the text columns (`Name`, `Sex`, …), which have no correlation.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(train_raw.drop(columns="PassengerId").corr(numeric_only=True),
            vmin=-1, vmax=1, cmap="coolwarm", annot=True, fmt=".2f", square=True)
plt.show()

`Pclass` (−0.34) and `Fare` (+0.26) are the numeric columns most related to `Survived`.
Correlation only captures straight-line relationships, and `Sex` is not even included yet,
so this is a starting point, not a verdict.

## D. Feature engineering

We now look at each variable and decide how to turn it into a numeric feature. The
transformations in this section are **explorations** on a copy called `eda`; section E
gathers the ones we keep into a single function that we apply to both files.

The bar charts below show the survival **rate** per group. The black line on each bar is a
95% confidence interval that seaborn computes **by bootstrap**, as in notebook 02.

In [ ]:
eda = train_raw.copy()

### Feature 1: `Pclass`

In [ ]:
sns.barplot(data=eda, x="Pclass", y="Survived")
plt.ylabel("Survival rate")
plt.show()

63% of first-class passengers survived, against 47% in second class and 24% in third.

`Pclass` is stored as 1, 2, 3. A model could read this as a quantity ("class 3 is three times
class 1"). **One-hot encoding** avoids that: one 0/1 column per class.

In [ ]:
pclass_dummies = pd.get_dummies(eda["Pclass"], prefix="Pclass", dtype=int)
display(pclass_dummies.head())

### Feature 2: `Sex`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.barplot(data=eda, x="Sex", y="Survived", ax=axes[0])
axes[0].set_ylabel("Survival rate")
sns.countplot(data=eda, x="Sex", hue="Survived", palette="RdBu", ax=axes[1])
plt.tight_layout()
plt.show()

The strongest signal in the data: 74% of women survived, against 19% of men ("women
and children first"). Models need numbers, so we map the two values to 1 and 0 with `map`.
Because `map` returns a **new** column, we assign it back explicitly.

In [ ]:
eda["Sex"] = eda["Sex"].map({"male": 1, "female": 0})
display(eda["Sex"].value_counts())

### Feature 3: `Cabin`
The first letter of a cabin number is the **deck**. Missing cabins are shown
as `"?"` in this quick look.

In [ ]:
eda["Deck"] = eda["Cabin"].str[0].fillna("?")
sns.barplot(data=eda, x="Deck", y="Survived", order=sorted(eda["Deck"].unique()))
plt.ylabel("Survival rate")
plt.show()

Some decks have very few passengers (look at the width of their confidence intervals).
The clearest pattern is simpler: passengers **with** a recorded cabin survived much more
often (67%) than those without (30%), probably because cabins were recorded mostly for
first-class passengers. We keep a 0/1 feature: *is a cabin recorded?*

In [ ]:
eda["Cabin"] = eda["Cabin"].notna().astype(int)
sns.barplot(data=eda, x="Cabin", y="Survived")
plt.xlabel("Cabin recorded")
plt.ylabel("Survival rate")
plt.show()

### Feature 4: `Age`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
bins = range(0, 82, 2)
sns.histplot(eda.loc[eda["Survived"] == 1, "Age"], bins=bins, kde=True, color="tab:blue", ax=axes[0])
axes[0].set_title("Survived")
sns.histplot(eda.loc[eda["Survived"] == 0, "Age"], bins=bins, kde=True, color="tab:red", ax=axes[1])
axes[1].set_title("Died")
plt.tight_layout()
plt.show()

The relationship between age and survival is not a straight line: the youngest children
did best, and the effect fades quickly. **Age groups** let a model capture that. First, we
must decide what to do with the 177 missing ages.

### Exercise Q1 · Imputation
The cell below replaces every missing age with the placeholder **−0.5**, which then lands in
its own "unknown" age group. That keeps the information "age was missing" but throws away
everything else we know about these passengers.

**Your task:** replace the placeholder with a better estimate. The mean age is a start;
better, the **title** in `Name` (*Master* was used for young boys, *Miss* for unmarried
women, *Mrs* for married women) tells us a lot about a passenger's age. Then think about how
you would plug your method into the `preprocess` function of section E.

In [ ]:
## Your turn (Q1): replace this with a better technique.
eda["Age"] = eda["Age"].fillna(-0.5)

<details>
<summary>Q1 reference answer — open after trying</summary>

Extract the title, then fill each missing age with the **median age of passengers with the
same title**. The medians are computed on the **training file only**, then reused for the
test file, so that no information comes from the test set.

```python
def extract_title(names):
    """Return the title between the comma and the period, e.g. 'Mr' or 'Miss'."""
    titles = names.str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
    titles = titles.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    return titles.where(titles.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")


age_by_title = train_raw.groupby(extract_title(train_raw["Name"]))["Age"].median()
print(age_by_title)


def impute_age(data):
    """Fill missing ages with the median age of the passenger's title group."""
    return data["Age"].fillna(extract_title(data["Name"]).map(age_by_title))


eda["Age"] = impute_age(train_raw)
print("missing ages left:", eda["Age"].isna().sum())
```

The medians confirm the idea: about 3.5 years for *Master*, 21 for *Miss*, 30 for *Mr* and
35 for *Mrs*. To use it in the model, replace `data["Age"].fillna(-0.5)` in `preprocess`
(section E) with `impute_age(data)`. The "unknown" age group then disappears.

</details>

In [ ]:
age_labels = ["unknown", "0–5", "5–15", "15–25", "25–40", "40–60", "60+"]
eda["AgeGroup"] = pd.cut(eda["Age"], bins=[-1, 0, 5, 15, 25, 40, 60, 120], labels=age_labels)

sns.barplot(data=eda, x="AgeGroup", y="Survived")
plt.xlabel("Age group")
plt.ylabel("Survival rate")
plt.show()

Children under 5 have the highest survival rate. If you kept the −0.5 placeholder, the
"unknown" group survived less often than average: missing ages are more common in third
class.

### Feature 5: family
`SibSp + Parch + 1` gives the size of the family travelling together (the `+ 1` counts the
passenger).

In [ ]:
eda["FamilySize"] = eda["SibSp"] + eda["Parch"] + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=eda, x="FamilySize", y="Survived", ax=axes[0])
axes[0].set_ylabel("Survival rate")
sns.countplot(data=eda, x="FamilySize", ax=axes[1])
plt.tight_layout()
plt.show()

Most passengers travelled alone, and the large families are so rare that their rates are
unreliable (wide intervals). A simpler 0/1 feature, *travelling alone*, captures the main
pattern: 30% of passengers travelling alone survived, against 51% of the others.

In [ ]:
eda["IsAlone"] = (eda["FamilySize"] == 1).astype(int)
sns.barplot(data=eda, x="IsAlone", y="Survived")
plt.ylabel("Survival rate")
plt.show()

### Feature 6: `Embarked`

In [ ]:
display(eda.groupby("Embarked")["Survived"].agg(survival_rate="mean", passengers="count").round(3))

Passengers who boarded in Cherbourg survived more often (55%), probably because many of them
travelled in first class. The two missing ports will become `S`, the most frequent one.

### Features 7, 8, 9: `Fare`, `Name`, `Ticket`
These three columns are left for you.

### Exercise Q2 · Feature engineering
Build features from `Fare`, `Name` and `Ticket`, add them to the model's inputs, and check
whether the validation accuracy (section F) and the cross-validated accuracy (section H)
improve. Ideas:

- `Fare`: the distribution is very skewed; try `np.log1p`. Remember that one test passenger has no fare.
- `Name`: the title (see Q1), encoded as one-hot columns.
- `Ticket`: passengers who share a ticket number travelled together; how many people share each ticket?

In [ ]:
# Your turn (Q2): build features from Fare, Name and Ticket.

<details>
<summary>Q2 reference answer — open after trying</summary>

Run this **after** section E (it uses `preprocess`, `FEATURES` and `y_all`). Each new feature
is computed the same way for both files, and the values that are "learned" (the median fare)
come from the training file only.

```python
def extract_title(names):
    titles = names.str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
    titles = titles.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    return titles.where(titles.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")


fare_median = train_raw["Fare"].median()
# Ticket sizes use both files: a group can be split between them. No labels are used.
ticket_counts = pd.concat([train_raw["Ticket"], test_raw["Ticket"]]).value_counts()
TITLES = ["Mr", "Mrs", "Miss", "Master", "Rare"]


def preprocess_q2(raw):
    data = preprocess(raw)
    data["LogFare"] = np.log1p(raw["Fare"].fillna(fare_median))
    title = pd.Categorical(extract_title(raw["Name"]), categories=TITLES)
    data = data.join(pd.get_dummies(title, prefix="Title", dtype=int).set_index(raw.index))
    data["TicketGroupSize"] = raw["Ticket"].map(ticket_counts)
    return data


FEATURES_Q2 = FEATURES + ["LogFare", "TicketGroupSize"] + [f"Title_{t}" for t in TITLES]
train_q2 = preprocess_q2(train_raw)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
for name, cols in [("original features", FEATURES), ("with Q2 features", FEATURES_Q2)]:
    scores = cross_val_score(forest, train_q2[cols], y_all, cv=cv)
    print(f"{name:18s}: {scores.mean():.3f} ± {scores.std():.3f}")
```

With a random forest, the new features raise the cross-validated accuracy by about two points.
The fixed list of title categories guarantees that both files get the same columns even if a
title is absent from one of them.

</details>

## E. Preprocessing: one function for both files

Section D explored features on a copy. Now we gather the transformations we keep into a
function. Applying **the same function** to the training and test files guarantees that the
model sees the same columns, encoded the same way, at training and prediction time.

The function starts from the raw data each time and returns a new table; it never modifies
its input.

In [ ]:
AGE_BINS = [-1, 0, 5, 15, 25, 40, 60, 120]


def preprocess(raw):
    """Turn a raw Titanic table into numeric features. Returns a new DataFrame."""
    data = raw.copy()
    data["Sex"] = data["Sex"].map({"male": 1, "female": 0})
    data = data.join(pd.get_dummies(data["Pclass"], prefix="Pclass", dtype=int))
    data["Cabin"] = data["Cabin"].notna().astype(int)
    # Q1: replace the -0.5 placeholder with a better imputation.
    data["Age"] = pd.cut(data["Age"].fillna(-0.5), bins=AGE_BINS, labels=False)
    data["Embarked"] = data["Embarked"].fillna("S").map({"C": 1, "Q": 2, "S": 3})
    data["FamilySize"] = data["SibSp"] + data["Parch"] + 1
    data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
    return data


train = preprocess(train_raw)
test = preprocess(test_raw)

FEATURES = ["Sex", "Age", "Cabin", "Pclass_1", "Pclass_2", "Pclass_3", "Embarked", "IsAlone"]
# After Q2, add your new features to this list (e.g. "LogFare", "Title_Mr", ...).
TARGET = "Survived"

X_all, y_all = train[FEATURES], train[TARGET]
display(X_all.head())
print("missing values in the training features:", int(X_all.isna().sum().sum()))
print("missing values in the test features:    ", int(test[FEATURES].isna().sum().sum()))

Every feature is now a number and nothing is missing. Two shortcuts remain that you can
improve in Q5. `Age` is now an age-group number from 0 to 6. `Embarked` is coded 1/2/3,
which implies an order between ports that does not exist (one-hot encoding would be more
faithful).

### Exercise Q3 · PCA
(a) Apply PCA to these features (try different subsets) and plot the passengers in 2-D,
coloured by `Survived`. Do the two classes separate?

(b) Keep the components that explain **90% of the variance**, fit a logistic regression on
them, and compare its cross-validated accuracy with a logistic regression on the original
features.

Is PCA a good choice for this problem?

In [ ]:
# Your turn (Q3): PCA in 2-D, then a logistic regression on 90% of the variance.

<details>
<summary>Q3 reference answer — open after trying</summary>

The features have different scales (`Age` goes from 0 to 6, the others are 0/1 or 1–3), so we
standardise them before PCA; otherwise the widest feature dominates the components.

```python
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# (a) 2-D view
coords = make_pipeline(StandardScaler(), PCA(n_components=2)).fit_transform(X_all)
plt.scatter(coords[:, 0], coords[:, 1], c=y_all, cmap="coolwarm", alpha=0.6)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Passengers on the first two components (red = survived)")
plt.show()

# (b) logistic regression with and without PCA
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
with_pca = make_pipeline(StandardScaler(), PCA(n_components=0.90), LogisticRegression(max_iter=1000))
without_pca = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
for name, model in [("with PCA (90%)", with_pca), ("without PCA", without_pca)]:
    scores = cross_val_score(model, X_all, y_all, cv=cv)
    print(f"{name:15s}: {scores.mean():.3f} ± {scores.std():.3f}")
print("components kept:", with_pca.fit(X_all, y_all).named_steps["pca"].n_components_)
```

The 2-D plot shows overlapping groups of points (the features take only a few values), with
survivors concentrated in some groups. Keeping 90% of the variance needs most of the
components, and the accuracy is about the same as without PCA. PCA is not very useful here:
there are only 8 features, so there is little to compress, and the components are harder to
interpret than "sex" or "class". PCA pays off with many correlated features, such as the 64
pixels of notebook 01.

</details>

## F. A first model

As in notebook 01, we keep part of the training file aside to **validate** our choices:
70% to train, 30% to validate. `stratify=y_all` keeps the same survival rate in both parts,
and `random_state` makes the split reproducible.

(The Kaggle test file cannot play this role, since we do not know its answers.)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.3, stratify=y_all, random_state=76
)

first_tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
print(f"decision tree, validation accuracy: {first_tree.score(X_val, y_val):.3f}")

Close to 80%, well above the 62% baseline. Can other models do better?

## G. Comparing classifiers

We train eight classifiers with reasonable settings and compare their accuracy on the
training part and on the validation part.

| Model | Idea in one line |
|---|---|
| Decision tree | A sequence of yes/no questions on the features |
| Logistic regression | A weighted sum of the features turned into a probability |
| k-nearest neighbours | Predict like the *k* most similar training passengers |
| SVM | The boundary with the widest margin between the classes |
| Perceptron | The simplest linear classifier, ancestor of neural networks |
| Random forest | Many trees on bootstrap samples; majority vote |
| Extra trees | Like a random forest, with more randomly chosen splits |
| AdaBoost | Many small trees, each focusing on the previous ones' mistakes |

`class_weight="balanced"` gives the less frequent class (survivors) more weight during
training. `random_state` makes the randomised models reproducible.

In [ ]:
models = {
    "Decision tree": DecisionTreeClassifier(class_weight="balanced", min_weight_fraction_leaf=0.01, random_state=0),
    "Logistic regression": LogisticRegression(max_iter=1000),
    "k-nearest neighbours": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(random_state=0),
    "Perceptron": Perceptron(class_weight="balanced", random_state=0),
    "Random forest": RandomForestClassifier(n_estimators=100, random_state=0),
    "Extra trees": ExtraTreesClassifier(n_estimators=300, max_features="sqrt", bootstrap=True,
                                        min_samples_split=10, random_state=0),
    "AdaBoost": AdaBoostClassifier(n_estimators=400, learning_rate=0.1, random_state=0),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append({
        "Classifier": name,
        "Training accuracy (%)": round(100 * model.score(X_train, y_train), 1),
        "Validation accuracy (%)": round(100 * model.score(X_val, y_val), 1),
    })

results = pd.DataFrame(rows).sort_values("Validation accuracy (%)", ascending=False)
display(results)

How to read this table:

- All models land between 78% and 82% on validation. With 268 validation passengers, one
  passenger is 0.4 points, so differences of one or two points are within the noise we
  measured in notebooks 01 and 02. Do not conclude that the top model is the best one.
- A model much better on training than on validation is **overfitting**: it memorised details
  of the training passengers that do not generalise. Compare the random forest's two columns.
- k-NN, SVM, logistic regression and the perceptron are sensitive to the **scale** of the
  features. Ours are small integers, so it matters little here, but standardising them is a
  good habit (a hint for Q5).

## H. Ensembles and cross-validation

### Voting
A `VotingClassifier` asks several models and returns the **majority vote** (`voting="hard"`).
When models make different mistakes, the vote can be more accurate than each of them.
`clone` creates fresh, untrained copies of our models.

In [ ]:
voting = VotingClassifier(
    estimators=[
        ("knn", clone(models["k-nearest neighbours"])),
        ("svm", clone(models["SVM"])),
        ("tree", clone(models["Decision tree"])),
        ("extra", clone(models["Extra trees"])),
        ("logistic", clone(models["Logistic regression"])),
        ("adaboost", clone(models["AdaBoost"])),
    ],
    voting="hard",
)
voting.fit(X_train, y_train)
print(f"voting ensemble, validation accuracy: {voting.score(X_val, y_val):.3f}")

### A more reliable estimate
The validation score above depends on one particular split. Cross-validation over the whole
training file gives a steadier estimate (notebook 01).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = cross_val_score(voting, X_all, y_all, cv=cv, scoring="accuracy")
print("fold accuracies:", np.round(cv_scores, 3))
print(f"voting ensemble, cross-validated accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

The cross-validated accuracy is lower than the single validation score above: that split was
a little lucky. **Report the cross-validated number.**

## I. Predicting for Kaggle

For the final predictions, we retrain the ensemble on **all** 891 training passengers, then
predict the 418 test passengers. Kaggle expects a CSV with exactly two columns,
`PassengerId` and `Survived`.

In [ ]:
voting.fit(X_all, y_all)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": voting.predict(test[FEATURES]),
})

output_dir = ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
submission_path = output_dir / "titanic_submission.csv"
submission.to_csv(submission_path, index=False)

print(submission.shape, "→", submission_path)
display(submission.head())
print("predicted survival rate:", round(submission["Survived"].mean(), 3))

**Submitting is optional.** To get a score, create a free Kaggle account, join the
[Titanic competition](https://www.kaggle.com/c/titanic), and upload this file under
*Submit Prediction*. In past editions of this lab, a similar submission scored around 76%.
Kaggle scores a different set of passengers, so expect your score to differ somewhat from
your cross-validated accuracy.

### Exercise Q4 · Randomized search
Choose one of the algorithms above and use `RandomizedSearchCV` (notebook 01) to find good
hyperparameters. Use cross-validation on `X_all, y_all`. How much does the cross-validated
accuracy improve over the default settings?

In [ ]:
# Your turn (Q4): tune one model with RandomizedSearchCV.

<details>
<summary>Q4 reference answer — open after trying</summary>

A random forest has several hyperparameters that interact, which suits random search.

```python
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "n_estimators": randint(100, 500),
    "max_depth": [3, 4, 5, 6, 8, 10, None],
    "min_samples_leaf": randint(1, 10),
    "max_features": ["sqrt", "log2", None],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=0), param_distributions,
    n_iter=20, cv=cv, scoring="accuracy", random_state=0, n_jobs=-1,
)
search.fit(X_all, y_all)

default_scores = cross_val_score(RandomForestClassifier(random_state=0), X_all, y_all, cv=cv)
print(f"default random forest: {default_scores.mean():.3f}")
print(f"tuned random forest:   {search.best_score_:.3f}")
print("best parameters:", search.best_params_)
```

The tuned forest prefers shallower trees or larger leaves than the default, which overfits.
The gain is a point or two: tuning helps, but better **features** (Q2, Q5) usually help more.
`best_score_` is slightly optimistic because it is the best of 20 tries on the same folds.

</details>

### Exercise Q5 · Do better
Use everything from this demo (feature engineering, encoding, scaling, model choice, tuning)
to improve the cross-validated accuracy as much as you can. If you submit to Kaggle, aim for
**80% or more** on the leaderboard.

Hints:
1. Use one-hot encoding for the categorical features (`Embarked`, title, deck…).
2. Standardise the features for the scale-sensitive models.
3. Check which features are most related to the target, and drop the useless ones.
4. Use domain knowledge to extract information (titles, family groups, fare per person…).
5. Try stacking: `StackingClassifier` trains a model on the predictions of other models.

Put the preprocessing in a `Pipeline` with a `ColumnTransformer`, so that cross-validation
stays honest (notebook 01, section E).

In [ ]:
# Your turn (Q5): build the best model you can and report its cross-validated accuracy.

<details>
<summary>Q5 reference answer (one possible approach) — open after trying</summary>

Keep a few readable features, impute age by title, one-hot encode the categories, and let a
`ColumnTransformer` scale the numeric columns inside a pipeline.

```python
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


def extract_title(names):
    titles = names.str.extract(r",\s*([^.]+)\.", expand=False).str.strip()
    titles = titles.replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    return titles.where(titles.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")


age_by_title = train_raw.groupby(extract_title(train_raw["Name"]))["Age"].median()
fare_median = train_raw["Fare"].median()
ticket_counts = pd.concat([train_raw["Ticket"], test_raw["Ticket"]]).value_counts()


def build_features(raw):
    data = pd.DataFrame(index=raw.index)
    data["Title"] = extract_title(raw["Name"])
    data["Pclass"] = raw["Pclass"].astype(str)
    data["Sex"] = raw["Sex"]
    data["Embarked"] = raw["Embarked"].fillna("S")
    data["Age"] = raw["Age"].fillna(data["Title"].map(age_by_title))
    data["LogFare"] = np.log1p(raw["Fare"].fillna(fare_median))
    data["FamilySize"] = raw["SibSp"] + raw["Parch"] + 1
    data["TicketGroupSize"] = raw["Ticket"].map(ticket_counts)
    data["HasCabin"] = raw["Cabin"].notna().astype(int)
    return data


categorical = ["Title", "Pclass", "Sex", "Embarked"]
numeric = ["Age", "LogFare", "FamilySize", "TicketGroupSize", "HasCabin"]
encode = ColumnTransformer([
    ("categories", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numbers", StandardScaler(), numeric),
])

X_q5, X_q5_test = build_features(train_raw), build_features(test_raw)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
candidates = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(random_state=0),
    "gradient boosting": GradientBoostingClassifier(max_depth=3, n_estimators=100, random_state=0),
}
for name, model in candidates.items():
    scores = cross_val_score(make_pipeline(encode, model), X_q5, y_all, cv=cv)
    print(f"{name:20s}: {scores.mean():.3f} ± {scores.std():.3f}")
```

These pipelines reach about 82–83% in cross-validation, against about 79% for the ensemble
of section H. To submit, fit the best pipeline on `X_q5, y_all` and predict `X_q5_test`.
Leaderboard scores on this competition are usually a few points below cross-validated
accuracy, so 80% on Kaggle is a demanding target.

</details>

### Exercise Q6 · How sure are we?
For the final voting ensemble, estimate a **95% bootstrap confidence interval** for the
accuracy, using the training file only (notebook 02: train on a bootstrap sample, score on
the out-of-bag passengers). If you submitted to Kaggle, does your leaderboard score fall
inside the interval?

In [ ]:
# Your turn (Q6): bootstrap confidence interval for the voting ensemble.

<details>
<summary>Q6 reference answer — open after trying</summary>

100 repetitions keep the run time short; more repetitions give a smoother interval.

```python
rng = np.random.default_rng(0)
n = len(y_all)
boot_scores = []
for _ in range(100):
    idx = rng.choice(n, size=n, replace=True)
    oob = np.setdiff1d(np.arange(n), idx)
    model = clone(voting).fit(X_all.iloc[idx], y_all.iloc[idx])
    boot_scores.append(accuracy_score(y_all.iloc[oob], model.predict(X_all.iloc[oob])))

low, high = np.percentile(boot_scores, [2.5, 97.5])
print(f"mean OOB accuracy: {np.mean(boot_scores):.3f}")
print(f"95% bootstrap CI: [{low:.1%}, {high:.1%}]")
```

The interval is roughly 76% to 83%. A Kaggle score around 76–78% sits near its lower end. Two reasons explain a slightly lower leaderboard score: the test passengers are a
different sample, and our modelling choices (features, models) were made while looking at
the training data, which makes training-based estimates a little optimistic.

</details>

## Takeaways

- Start with the **question** and the **data**: the target, the features, the missing values, and a simple baseline to beat.
- Exploration guides **feature engineering**: the strongest signals here are sex, class, age group and whether a cabin is recorded.
- Put the transformations in **one function** (or better, a pipeline) and apply it to every file.
- Compare models on validation data, then report a **cross-validated** score, ideally with a **confidence interval**.
- Gains usually come more from better features than from more complex models.

## References

1. [Titanic competition on Kaggle](https://www.kaggle.com/c/titanic)
2. [Predicting the survival of Titanic passengers](https://towardsdatascience.com/predicting-the-survival-of-titanic-passengers-30870ccc7e8), Towards Data Science
3. [Titanic survival dataset: exploratory data analysis](https://becominghuman.ai/titanic-survival-dataset-part-1-2-exploratory-data-analysis-5b98f7917913), Becoming Human
4. [How to (almost) win Kaggle competitions](https://yanirseroussi.com/2014/08/24/how-to-almost-win-kaggle-competitions/), Yanir Seroussi
5. [Kaggle tips and tricks](https://www.dataquest.io/blog/kaggle-tips-tricks/), Dataquest
6. scikit-learn user guide: [ensemble methods](https://scikit-learn.org/stable/modules/ensemble.html) and [`ColumnTransformer`](https://scikit-learn.org/stable/modules/compose.html#column-transformer)